# Week 1 Experiment Notebook — LB-DSL-CO T2D Prototype
**Iteration 1: Grammar Definition, Forward-Chaining Engine & CLI Execution**

| Field | Value |
|---|---|
| Author | Zhenshuo Sun (zsun30@lakeheadu.ca) |
| Supervisors | Prof. Sabah Mohammed, Prof. Jinan Fiaidhi |
| Date | June 7, 2026 |
| Repo | https://github.com/firstglimmerofhope/lb-dsl-co-clinical-orders |
| Scope | Type 2 Diabetes (ICD-10: E11%) — Single Clinical Pathway |


## 1. Iteration 1 Objectives

This week's goal was to establish the minimal working prototype for the LB-DSL-CO system, covering:

1. Define a formal DSL grammar (`t2d-orders.langium`) using Langium for T2D clinical orders
2. Implement a Forward-Chaining execution engine in the CLI generator
3. Integrate Langium validator for grammatical/semantic constraint checking
4. Execute the CLI successfully on at least two representative patient cases
5. Validate that safety rules (eGFR-based Metformin contraindication) trigger correctly


## 2. Scope Constraints (Per Supervisor Guidance)

- **Disease**: Type 2 Diabetes ONLY (ICD-10: E11%)
- **Pathway**: Single pathway — Diagnosis → Medication Selection → Safety Check
- **Terminology**: Local JSON cache only (no real-time RxNorm/LOINC REST calls in prototype)
- **Datasets**: Pima Indians dataset for DSL ground truth (MIMIC-IV access pending CITI training)
- **Visualization**: Deferred to later Iteration (Sprotty graph, max 2 branch levels)
- **Clinical Disclaimer**: This is a **research prototype**, NOT a clinical decision support system. All LLM-generated content requires DSL Validator verification before use.


## 3. Forward-Chaining Execution Semantics (Pseudocode)

> **Design principle**: Pseudocode is defined FIRST; grammar is derived from it — not the reverse.

```
FUNCTION ForwardChain(patient: PatientData, rules: DiagnosisRule[], safetyRules: SafetyRule[]):

  // Phase 1: Diagnosis Inference
  diagnosis ← 'Undetermined'
  FOR EACH rule IN rules:
    IF rule.condition matches patient data:
      diagnosis ← rule.conclusion
      BREAK  // first-match semantics

  // Phase 2: Medication Selection (only if T2D diagnosed)
  medications ← []
  IF diagnosis == 'T2D':
    IF patient.a1c >= 8.5 OR patient.fpg >= 11.1:
      medications.append('Immediate Insulin / Combo Therapy')
    ELSE:
      medications.append('Metformin First-Line')  // default
      IF 'CKD' IN patient.comorbidities OR 'HF' IN patient.comorbidities:
        medications.append('SGLT2 Inhibitor — Cardiorenal Protection')
      ELIF 'ASCVD' IN patient.comorbidities:
        medications.append('GLP-1 Receptor Agonist — CV Protection')

  // Phase 3: Safety Rule Evaluation (runs AFTER medication selection)
  safetyAlerts ← []
  FOR EACH rule IN safetyRules:
    IF rule.condition matches patient.egfr:
      safetyAlerts.append(rule.alert)
      IF rule.action == 'REMOVE_METFORMIN':
        medications ← medications.filter(m => m != 'Metformin First-Line')

  RETURN { diagnosis, medications, safetyAlerts }
```


## 4. DSL Grammar (`t2d-orders.langium`)

The grammar was derived directly from the forward-chaining pseudocode above.

```langium
grammar T2DOrders

entry ClinicalModel:
    patients+=PatientData*
    rules+=DiagnosisRule*
    safetyRules+=SafetyRule*;

PatientData:
    'patient' name=ID '{'
        'a1c'    ':' a1c=NUMBER
        'fpg'    ':' fpg=NUMBER
        'egfr'   ':' egfr=NUMBER
        ('comorbidities' ':' '[' comorbidities+=ID (',' comorbidities+=ID)* ']')?
    '}';

DiagnosisRule:
    'rule' name=ID '{'
        'if'   condition=RuleCondition
        'then' conclusion=STRING
    '}';

SafetyRule:
    'safety' name=ID '{'
        'if'    condition=SafetyCondition
        'alert' alertText=STRING
    '}';

RuleCondition:
    field=('a1c'|'fpg') op=('>'|'>='|'<'|'<=') threshold=NUMBER;

SafetyCondition:
    'egfr' op=('<'|'<='|'>'|'>=') threshold=NUMBER;

hidden terminal WS: /\s+/;
terminal ID: /[_a-zA-Z][\w]*/;
terminal NUMBER returns number: /[0-9]+(\.[0-9]+)?/;
terminal STRING: /"[^"]*"/;
```


## 5. Test Case: `case01.t2d`

Two patients covering the two primary branching scenarios in the T2D pathway.

```
patient JohnDoe {
  a1c:   7.8
  fpg:   8.2
  egfr:  85
  comorbidities: [HF]
}

patient JaneSmith {
  a1c:   8.1
  fpg:   9.0
  egfr:  25
  comorbidities: [CKD]
}

rule R1_T2D_A1C {
  if a1c >= 6.5
  then "T2D"
}

rule R2_Prediabetes {
  if a1c >= 5.7
  then "Prediabetes"
}

safety S1_Metformin_Contraindicated {
  if egfr < 30
  alert "[CRITICAL] eGFR {value}: Metformin contraindicated — remove from order set"
}

safety S2_Metformin_DoseLimit {
  if egfr < 45
  alert "[WARNING] eGFR {value}: Limit Metformin to max 1000mg/day"
}
```


## 6. CLI Execution Output

**Command:**
```bash
node packages/cli/bin/cli.js run examples/case01.t2d
```

**Output:**
```
=== LB-DSL-CO T2D Forward-Chaining Report ===
Source: examples/case01.t2d

── Patient: JohnDoe ──
   A1C: 7.8%  |  FPG: 8.2 mmol/L  |  eGFR: 85
   Comorbidities: HF
   → Diagnosis:  T2D
   → Medications:
       • Metformin (First-Line)
       • SGLT2 Inhibitor (e.g., Empagliflozin) — Cardiorenal Protection

── Patient: JaneSmith ──
   A1C: 8.1%  |  FPG: 9 mmol/L  |  eGFR: 25
   Comorbidities: CKD
   → Diagnosis:  T2D
   → Medications:
       • SGLT2 Inhibitor (e.g., Empagliflozin) — Cardiorenal Protection
       • Alternative non-renal-cleared therapy required
   → Safety Alerts:
       ⚠ [CRITICAL] eGFR 25: Metformin contraindicated — remove from order set
       ⚠ [WARNING] eGFR 25: Limit Metformin to max 1000mg/day
```


## 7. Result Analysis

### 7.1 Correctness Verification

| Patient | Expected Diagnosis | Actual | Expected Key Medication | Actual | Safety Alert Expected | Actual |
|---|---|---|---|---|---|---|
| JohnDoe (A1C=7.8, eGFR=85, HF) | T2D | ✅ T2D | Metformin + SGLT2 (HF branch) | ✅ | None | ✅ None |
| JaneSmith (A1C=8.1, eGFR=25, CKD) | T2D | ✅ T2D | SGLT2 only (Metformin removed) | ✅ | CRITICAL + WARNING | ✅ Both fired |

### 7.2 Clinical Rule Basis

- **Metformin contraindication at eGFR < 30**: Consistent with ADA Standards of Care 2024 (Section 9) and Diabetes Canada CPG 2023 (Chapter 13). At eGFR 25, Metformin accumulates and risks lactic acidosis.
- **SGLT2 Inhibitor for CKD/HF patients**: Consistent with ADA recommendation for cardiorenal risk patients — Empagliflozin and Dapagliflozin are first-line additions regardless of A1C level.
- **eGFR 45 warning for dose limiting**: Consistent with Diabetes Canada guideline (max 1000mg/day when 30 ≤ eGFR < 45).

### 7.3 Forward-Chaining Chain Trace (JaneSmith)

```
INPUT:  a1c=8.1, fpg=9.0, egfr=25, comorbidities=[CKD]
STEP 1: R1_T2D_A1C fires (a1c=8.1 >= 6.5) → diagnosis = 'T2D'
STEP 2: Medication selection — T2D confirmed
        a1c < 8.5 and fpg < 11.1 → no immediate insulin
        Metformin added (default first-line)
        CKD in comorbidities → SGLT2 Inhibitor added
STEP 3: Safety evaluation
        S1 fires (egfr=25 < 30) → CRITICAL alert, Metformin REMOVED
        S2 fires (egfr=25 < 45) → WARNING alert
OUTPUT: diagnosis=T2D, medications=[SGLT2, Alt-therapy], alerts=[CRITICAL, WARNING]
```


## 8. Technical Issues Encountered & Resolutions

| Issue | Root Cause | Resolution |
|---|---|---|
| `ERR_MODULE_NOT_FOUND` for `language/src/*.js` | Import path pointed to `src/` not compiled `out/` | Fixed all imports to reference `out/` directory |
| `SyntaxError: no export named 'default'` | `bin/cli.js` used `import main from ...` but `main.ts` has no default export | Changed `bin/cli.js` to `import '../out/main.js'` (side-effect import) |
| `ServiceRegistry is empty` | Language services not registered before document parsing | Added `services.shared.ServiceRegistry.register(services)` before `extractDocument` |
| `AstNode not assignable to ClinicalModel` | `parseResult.value` typed as `AstNode` | Added `isClinicalModel()` type guard from generated AST |


## 9. Pima Indians Dataset — Preliminary DSL Ground Truth Mapping

The Pima dataset (Kaggle/UCI) provides: `Glucose`, `BMI`, `BloodPressure`, `Age`, `Outcome` (0/1).
Mapping to DSL fields (approximate — for ground truth generation only):

| Pima Feature | DSL Field | Unit Conversion |
|---|---|---|
| Glucose (mg/dL) | `fpg` (mmol/L) | ÷ 18.0 |
| HbA1c (if available) | `a1c` | direct % |
| Outcome=1 | Expected: `T2D` diagnosis | — |
| Outcome=0 | Expected: `Normal` or `Prediabetes` | — |

> **Note**: Pima dataset does not include eGFR or comorbidities. Safety rule evaluation will use synthetic eGFR values for ground truth generation in later Iteration. MIMIC-IV (pending CITI credentialing) will be used for real clinical order validation.


In [1]:
# Preliminary: Load Pima dataset and map to DSL-compatible fields
# (Requires: pip install pandas scikit-learn)

import pandas as pd

# Load dataset (download from: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
# df = pd.read_csv('pima-indians-diabetes.csv')

# Simulated sample for notebook demonstration
data = {
    'Glucose': [148, 85, 183, 89, 137],
    'BMI': [33.6, 26.6, 23.3, 28.1, 43.1],
    'Age': [50, 31, 32, 21, 33],
    'Outcome': [1, 0, 1, 0, 1]
}
df = pd.DataFrame(data)

# Convert Glucose mg/dL → mmol/L (approximate FPG)
df['fpg_mmol'] = (df['Glucose'] / 18.0).round(1)

# Apply DSL diagnosis rule: FPG >= 7.0 → T2D
df['dsl_diagnosis'] = df['fpg_mmol'].apply(
    lambda x: 'T2D' if x >= 7.0 else ('Prediabetes' if x >= 5.6 else 'Normal')
)

# Compare with Pima ground truth
df['pima_label'] = df['Outcome'].map({1: 'T2D', 0: 'Non-T2D'})
df['match'] = df.apply(
    lambda r: '✅' if (r['dsl_diagnosis'] == 'T2D') == (r['pima_label'] == 'T2D') else '❌', axis=1
)

print(df[['Glucose', 'fpg_mmol', 'dsl_diagnosis', 'pima_label', 'match']].to_string(index=False))

 Glucose  fpg_mmol dsl_diagnosis pima_label match
     148       8.2           T2D        T2D     ✅
      85       4.7        Normal    Non-T2D     ✅
     183      10.2           T2D        T2D     ✅
      89       4.9        Normal    Non-T2D     ✅
     137       7.6           T2D        T2D     ✅


## 10. Iteration 2 Plan

Based on supervisor feedback, the following grammar extensions are planned:

### Grammar additions:
```langium
MedicationOrder:
    'medication' name=ID '{'
        'drug'      ':' drug=STRING
        'dose'      ':' dose=NUMBER
        'unit'      ':' unit=STRING
        'route'     ':' route=RouteEnum
        'frequency' ':' frequency=STRING
        ('indication' ':' indication=STRING)?
    '}';

LabOrder:
    'lab' name=ID '{'
        'test'      ':' test=STRING
        ('specimen'  ':' specimen=STRING)?
        'priority'  ':' priority=PriorityEnum
        ('indication' ':' indication=STRING)?
    '}';

RouteEnum returns string: 'PO' | 'IV' | 'PR' | 'SC';
PriorityEnum returns string: 'STAT' | 'Routine' | 'TimedAM';
```

### Validator additions (Iteration 2 focus):
- `dose` within clinically plausible range (e.g., Metformin: 500–2000mg)
- `route` must be a valid FDA-recognized administration route
- `priority=STAT` requires `indication` to be non-empty
- Local RxNorm JSON cache lookup for drug name validation

### Paper draft update due:
- Section 3 (Methodology): Forward-chaining semantics + grammar design rationale
- Section 4 (Implementation): Langium toolchain, monorepo structure


---
## ⚠️ Research Prototype Disclaimer

This system is a **research prototype** developed for the IEEE paper submission. It is scoped exclusively to Type 2 Diabetes (ICD-10: E11%) as a single clinical pathway demonstration. It is **NOT** a clinical decision support system and must not be used for actual patient care decisions. All LLM-generated clinical content (planned for later Iteration VGCL component) requires DSL Validator verification before any use or evaluation.
